# 03a   Structure Extraction

**Pipeline position:** `01 -> 02 -> **03a** -> 03b`.

**Purpose:** derive per-document *structure maps*   structural article
headings (number + title + paragraph count), chapters/titles, recitals,
tables, obligations, energy units, and a chunking-strategy recommendation  
from the **shared processed markdown** (`data/processed/eu/**/*.md`), which
notebook 02 established should come from the best text parser
(`pymupdf4llm`, 1.0 structure score).

**Method:** deterministic regex only   no LLM calls, no cache, idempotent.
All extraction lives in `src/structure/structure_maps.py`; GT fields (title / instrument
/ tables / obligations) reuse the same helpers from `src/ground_truth.py`
that notebook 02 used, so *map vs GT* is a true data-quality check rather than
a detector-mismatch check.

**Key modelling decision:** an article is *structural* only if its number is
a heading (`## _Article 2_`, `## **Article 2 - Controller processing**`).
`Article 14 of Directive 2019/944 ...` inside prose is a **mention** and is
counted separately. This is what made the text parsers score 1.0 on
`structure_score` in 02.

**Outputs** (all under `notebooks/data/structure_maps/`):
- `{doc_id}_structure.json`   one map per document
- `structure_maps.json`   combined
- `structure_maps_summary.csv`   one row per doc
- `structure_coverage.csv`   map vs GT v2 audit
- `best_structure_map.json`   coverage-best map
- `structure_handoff.json`   hand-off for 03b


---
## 0   Setup

In [1]:

import os, sys, json, time, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
from rich.console import Console
from rich import print as rprint
from rich.progress import track

warnings.filterwarnings("ignore")
console = Console()

def _repo_root() -> Path:
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "src"))

import common as c            # paths, discovery
from parsing import ground_truth as GT     # shared regex helpers (title/instrument/tables/...)
from structure import structure_maps as SM   # <-- the 03a engine (this notebook just drives it)

OUT  = c.NOTEBOOKS_DATA / "structure_maps"
OUT.mkdir(parents=True, exist_ok=True)
GT_PATH     = c.GROUND_TRUTH_PATH / "ground_truth_all_docs_v2.json"
BEST_PARSER = c.EVALUATIONS_PATH / "best_parser.json"

rprint("[bold]root :[/bold]", "../" + ROOT.name)
rprint("[bold]out  :[/bold]", "../" + ROOT.name + "/notebooks/data")
rprint("[bold]best text parser (from 02):[/bold]",
       json.loads(BEST_PARSER.read_text())["best_text_parser"])


root : ../energy-compliance-auditout  : ../energy-compliance-audit/notebooks/data/structure_maps

---
## 1   Build all structure maps

Engine: `src/structure_maps.build_all_maps()`   walks `data/processed/eu/**/*.md`, extracts structure per doc, writes `{doc}_structure.json` each. Fast (regex only), so one pass over all 25 docs.

In [2]:

mds = sorted(c.discover_markdown())
rprint("[bold]%d[/bold] markdown files under [bold]%s[/bold]" % (len(mds), c.PROCESSED_BASE))

t0 = time.time()
maps = {}
for p in track(mds, description="structure maps"):
    smap = SM.build_structure_map(p).to_dict()
    maps[p.stem] = smap
    rprint("  %-28s arts=%-4d chap=%-3d rec=%-4d tab=%-3d obl=%-4d strat=%s" % (
        p.stem, smap["article_count"], len(smap["chapters"]),
        smap["recitals_count"], smap["tables"]["count"],
        smap["obligations"]["count"], smap["chunking_strategy"]))
rprint("[green]%d maps in %.1fs[/green]" % (len(maps), time.time() - t0))
assert len(maps) == len(mds), "every doc must produce a map"


25 markdown files under /home/iauser/AIProjects/energy-audit/data/processed/eu

Output()

elec_dir_2019_944            arts=74   chap=0   rec=162  tab=4   obl=629  strat=article_based

emd_reform_dir_2024_1711     arts=5    chap=0   rec=39   tab=0   obl=86   strat=article_based

eprivacy_dir_2002_58         arts=21   chap=0   rec=49   tab=0   obl=65   strat=article_based

...

metering_data_2023_1162      arts=14   chap=0   rec=35   tab=191 obl=44   strat=article_based

mica_2023_1114               arts=81   chap=6   rec=170  tab=0   obl=812  strat=hierarchical

remit_1227_2011              arts=21   chap=0   rec=46   tab=0   obl=131  strat=article_based

remit_ii_2024_1106           arts=18   chap=0   rec=52   tab=0   obl=296  strat=article_based

25 maps in 7.6s

---
## 2   Summary table + write outputs

In [3]:

rows = []
for doc_id, sm in maps.items():
    rows.append({
        "doc_id":                doc_id,
        "category":              sm["category"],
        "content_chars":         sm["content_chars"],
        "words":                 sm["words"],
        "article_count":         sm["article_count"],
        "chapter_count":         len(sm["chapters"]),
        "recitals_count":        sm["recitals_count"],
        "tables_count":          sm["tables"]["count"],
        "obligations":           sm["obligations"]["count"],
        "derogations":           sm["derogations"]["count"],
        "units_total":           sum(sm["units"].values()),
        "mean_para_per_article": sm["paragraph_stats"]["mean_per_article"],
        "chunking_strategy":     sm["chunking_strategy"],
        "document_title":        sm["document_title"],
        "instrument":            sm["instrument"],
        "publication_date":      str(sm["publication_date"]),
    })
df = pd.DataFrame(rows).sort_values(["category", "doc_id"])

# write combined + csv + best + handoff (best selected in section 3)
(OUT / "structure_maps.json").write_text(
    json.dumps({"generated": datetime.now().isoformat(),
                "generator": "03a (src/structure/structure_maps.py)",
                "docs": maps}, indent=2, ensure_ascii=False),
    encoding="utf-8")

rprint(df[["doc_id", "article_count", "chapter_count", "recitals_count",
          "tables_count", "obligations", "chunking_strategy"]]
       .sort_values("article_count", ascending=False).to_string(index=False))
rprint("\n[bold]strategy mix:[/bold]",
       {s: int(n) for s, n in Counter(r["chunking_strategy"] for r in rows).items()})


doc_id  article_count  chapter_count  recitals_count  tables_count  obligations   
chunking_strategy
         entso_sogl_2017_1485            192             27             201            19         1083        
hierarchical
                gdpr_2016_679             99              0             199             0          487       
article_based
         entso_cacm_2015_1222             84              5              35             0          542        
hierarchical
               mica_2023_1114             81              6             170             0          812        
hierarchical
            elec_dir_2019_944             74              0             162             4          629       
article_based
         entso_ncrfg_2016_631             72              7              96            46          647        
hierarchical
            elec_reg_2019_943             71              8             146             4          722        
hierarchical
         entso_ebgl_2017_2195             65             10              66             2          446        
hierarchical
               dora_2022_2554             64              0             177             0          455       
article_based
               eidas_2014_910             52              6             118             0          231        
hierarchical
          eu_ai_act_2024_1689             51              5             248             0          376        
hierarchical
           data_act_2023_2854             50             14             161             0          327        
hierarchical
           nis2_dir_2022_2555             46              9             185            27          327        
hierarchical
            eidas_2_2024_1183             27              0              99             0          356       
article_based
         eprivacy_dir_2002_58             21              0              49             0           65       
article_based
              remit_1227_2011             21              0              46             0          131       
article_based
           dlt_pilot_2022_858             19              0              82             0          199       
article_based
           remit_ii_2024_1106             18              0              52             0          296       
article_based
      metering_data_2023_1162             14              0              35           191           44       
article_based
     emd_reform_reg_2024_1747             12              0              73             0          163       
article_based
     emd_reform_dir_2024_1711              5              0              39             0           86       
article_based
  know_your_contract_guidance              0              0               0            10            2 
table_then_sentence
    entso-e_simulation_models              0             16               0            78           84 
table_then_sentence
entso-e_compliance_monitoring              0              3               0            12           35 
table_then_sentence
          acer_remit_guidance              0              0             381            63          128 
table_then_sentence

strategy mix:
{'article_based': 11, 'hierarchical': 10, 'table_then_sentence': 4}

---
## 3   Map vs GT coverage audit + hand-off

Compared against the 02 LLM-reviewed ground truth (`ground_truth_all_docs_v2.json`). Because both sides use the same `src/ground_truth.py` regexes, `title/instrument/tables` near-misses are real signal, not detector drift. `article_recall` = GT article numbers found among the doc's *structural* headings.

In [4]:

def _norm(t):
    if not t:
        return ""
    return " ".join(str(t).replace("*", "").split()).lower()

def _match(gt, got) -> int:
    if gt in (None, "", []):
        return 1
    a, b = _norm(gt), _norm(got)
    if not b:
        return 0
    return int(a == b or a in b or b in a)

gt_docs = json.loads(GT_PATH.read_text())["documents"]
cov_rows = []
for doc_id, sm in maps.items():
    g = gt_docs.get(doc_id, {}).get("global", {})
    gt_arts = set(str(x) for x in (g.get("article_numbers") or []))
    head_arts = {a["number"] for a in sm["articles"]}
    recall = (len(gt_arts & head_arts) / len(gt_arts)) if gt_arts else 1.0
    row = {
        "doc_id": doc_id,
        "title_ok": _match(g.get("document_title"), sm["document_title"]),
        "instrument_ok": _match(g.get("instrument"), sm["instrument"]),
        "date_ok": 1 if (not g.get("publication_date")
                         or str(g["publication_date"]) == str(sm.get("publication_date"))) else 0,
        "article_recall": round(recall, 3),
        "article_struct": len(head_arts),
        "article_gt": len(gt_arts),
        "tables_ok": int(abs((g.get("tables_count") or 0) - sm["tables"]["count"]) <=
                          max(2, (g.get("tables_count") or 0) * 0.1)),
    }
    row["coverage"] = round((row["title_ok"] + row["instrument_ok"] + row["date_ok"]
                             + row["article_recall"] + row["tables_ok"]) / 5, 3)
    cov_rows.append(row)

cov = pd.DataFrame(cov_rows).sort_values(["coverage", "doc_id"])
rprint("[bold]coverage:[/bold] min=%.3f  mean=%.3f  max=%.3f" %
       (cov["coverage"].min(), cov["coverage"].mean(), cov["coverage"].max()))
low = cov[cov["coverage"] < 0.8]
rprint("[bold]docs < 0.8:[/bold]", low["doc_id"].tolist() or "none")
console.print(cov.head(30).to_string(index=False))
cov.to_csv(OUT / "structure_coverage.csv", index=False)

# persist summary + coverage summary
df.to_csv(OUT / "structure_maps_summary.csv", index=False)

# best map by coverage (tie-break: more structural articles)
cov_nontrivial = cov[cov["article_struct"] > 0]
# best map: highest coverage among docs that actually have structure
# (a 0-article doc cannot drive an AST-based demo), ties -> more articles
best_id = (cov_nontrivial.sort_values(["coverage", "article_struct"],
                                      ascending=[False, False]).iloc[0]["doc_id"])
best = {"doc_id": best_id, **maps[best_id]}
(OUT / "best_structure_map.json").write_text(json.dumps(best, indent=2, ensure_ascii=False))

handoff = {
    "generated": datetime.now().isoformat(),
    "docs_mapped": len(maps),
    "strategy_mix": {s: int(n) for s, n in Counter(m["chunking_strategy"] for m in maps.values()).items()},
    "coverage": {
        "mean": round(float(cov["coverage"].mean()), 3),
        "min":  round(float(cov["coverage"].min()), 3),
        "docs_below_08": low["doc_id"].tolist(),
    },
    "best_by_coverage": best_id,
    "files": {
        "per_doc":  "notebooks/data/structure_maps/{doc_id}_structure.json",
        "combined": "notebooks/data/structure_maps/structure_maps.json",
        "summary":  "notebooks/data/structure_maps/structure_maps_summary.csv",
        "coverage": "notebooks/data/structure_maps/structure_coverage.csv",
        "best_map": "notebooks/data/structure_maps/best_structure_map.json",
    },
    "note": ("03b reads the chosen map + source_md and walks the article spans "
             "(line_start/span_end in src/structure/structure_maps.py) to build the AST; "
             "the chunking_strategy field picks the node granularity"),
}
(OUT / "structure_handoff.json").write_text(json.dumps(handoff, indent=2))

rprint("[bold]outputs:[/bold]")
for f in sorted(OUT.iterdir()):
    rprint("   ", f.name, "%6d" % len(f.read_bytes()), "bytes")


coverage: min=0.200  mean=0.698  max=1.000

docs < 0.8:
[
    'acer_remit_guidance',
    'eprivacy_dir_2002_58',
    'eidas_2014_910',
    'entso-e_compliance_monitoring',
    'entso-e_simulation_models',
    'emd_reform_dir_2024_1711',
    'emd_reform_reg_2024_1747',
    'remit_ii_2024_1106',
    'eidas_2_2024_1183',
    'dlt_pilot_2022_858',
    'metering_data_2023_1162',
    'eu_ai_act_2024_1689',
    'mica_2023_1114',
    'remit_1227_2011',
    'dora_2022_2554',
    'entso_ebgl_2017_2195',
    'nis2_dir_2022_2555',
    'data_act_2023_2854',
    'elec_dir_2019_944',
    'elec_reg_2019_943',
    'gdpr_2016_679'
]

doc_id  title_ok  instrument_ok  date_ok  article_recall  article_struct  article_gt  
tables_ok  coverage
          acer_remit_guidance         0              1        0           0.000               0          28        
0     0.200
         eprivacy_dir_2002_58         1              0        0           0.733              21          15        
1     0.547
               eidas_2014_910         1              0        0           0.900              52          30        
1     0.580
entso-e_compliance_monitoring         1              1        0           0.000               0          11        
1     0.600
    entso-e_simulation_models         1              1        0           0.000               0          23        
1     0.600
     emd_reform_dir_2024_1711         1              1        0           0.107               5          28        
1     0.621
     emd_reform_reg_2024_1747         1              1        0           0.175              12          40        
1     0.635
           remit_ii_2024_1106         1              1        0           0.226              18          31        
1     0.645
            eidas_2_2024_1183         1              1        0           0.286              27          63        
1     0.657
           dlt_pilot_2022_858         1              1        0           0.469              19          32        
1     0.694
      metering_data_2023_1162         1              1        0           0.500              14          14        
1     0.700
          eu_ai_act_2024_1689         1              1        0           0.529              51          87        
1     0.706
               mica_2023_1114         1              1        0           0.608              81         102        
1     0.722
              remit_1227_2011         1              1        0           0.680              21          25        
1     0.736
               dora_2022_2554         1              1        0           0.790              64          62        
1     0.758
         entso_ebgl_2017_2195         1              1        0           0.790              65          62        
1     0.758
           nis2_dir_2022_2555         1              1        0           0.808              46          52        
1     0.762
           data_act_2023_2854         1              1        0           0.865              50          37        
1     0.773
            elec_dir_2019_944         1              1        0           0.902              74          82        
1     0.780
            elec_reg_2019_943         1              1        0           0.910              71          78        
1     0.782
                gdpr_2016_679         1              1        0           0.926              99          68        
1     0.785
         entso_cacm_2015_1222         1              1        0           1.000              84          56        
1     0.800
         entso_ncrfg_2016_631         1              1        0           1.000              72          38        
1     0.800
         entso_sogl_2017_1485         1              1        0           1.000             192         130        
1     0.800
  know_your_contract_guidance         1              1        1           1.000               0           0        
1     1.000

outputs:

acer_remit_guidance_structure.json   2060 bytes

...

remit_ii_2024_1106_structure.json   3366 bytes

structure_coverage.csv   1213 bytes

structure_handoff.json   1493 bytes

structure_maps.json 250917 bytes

structure_maps_summary.csv   4918 bytes